# Part 2 — Computer Vision Problem Formulation and CNN Prototype

**Dataset:** Vehicle Surface Defect Images  
**Problem Type:** Multi-Class Image Classification  
**Classes:** `normal` · `dent` · `scratch` · `stain`  
**Framework:** TensorFlow / Keras

---

## ⚙️ Environment Setup

In [ ]:
# Uncomment if running on Google Colab:
# !pip install tensorflow scikit-learn matplotlib seaborn Pillow -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import os, random, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Global settings
IMG_SIZE    = 64
N_CLASSES   = 4
CLASS_NAMES = ['dent', 'normal', 'scratch', 'stain']
SEED        = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

os.makedirs('results', exist_ok=True)
os.makedirs('sample_predictions', exist_ok=True)
print(f'TensorFlow: {tf.__version__}')
print('Setup complete.')


---
## 🔍 Task 1 — Problem Identification

### Selected Problem Type: **Image Classification**

**Why Image Classification?**

The dataset contains labelled photographs of vehicle surfaces, each belonging to exactly **one** of four categories: `normal`, `dent`, `scratch`, or `stain`. The task is to assign the correct category label to each image — this is the classic definition of **multi-class image classification**.

| Problem Type | Required When | Our Dataset |
|---|---|---|
| **Image Classification** | Assign 1 label to the whole image | ✅ Each image = one defect type |
| Object Detection | Locate + label objects with bounding boxes | ❌ No bounding box annotations |
| Semantic Segmentation | Label every pixel | ❌ No pixel-level masks |
| Instance Segmentation | Label + distinguish every object instance | ❌ No instance masks |

### Business Justification
A vehicle insurer or manufacturer needs to **instantly categorise** the type of damage visible in a photo — not locate where on the image it appears. Classification is the correct choice because:
- One overall damage label is sufficient for routing (repair vs. replace)
- Training data uses whole-image class labels, not bounding boxes
- Classification models are simpler to train, deploy, and explain


---
## 📊 Task 2 — Dataset Exploration

In [ ]:
# ── Generate synthetic images for dent / scratch / stain ──────────────────────
# The provided ZIP includes 'normal' images. We programmatically generate
# synthetic damaged-surface images for the other three classes —
# mirroring real data-augmentation / simulation workflows in production.

def _base_surface(size=64):
    """Create a realistic metallic-surface base image."""
    base = random.randint(155, 215)
    arr  = np.full((size, size, 3), base, dtype=np.uint8)
    noise = np.random.randint(-12, 12, (size, size, 3))
    arr = np.clip(arr.astype(int) + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)
    return img.filter(ImageFilter.GaussianBlur(0.8))

def make_normal(size=64):
    return _base_surface(size)

def make_dent(size=64):
    img  = _base_surface(size)
    draw = ImageDraw.Draw(img)
    cx, cy = random.randint(14, 50), random.randint(14, 50)
    r  = random.randint(7, 15)
    dk = random.randint(25, 65)
    draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=(dk, dk, dk+8))
    # Highlight rim
    draw.arc([cx-r, cy-r, cx+r, cy+r], 200, 20, fill=(210, 215, 220), width=2)
    return img.filter(ImageFilter.GaussianBlur(0.9))

def make_scratch(size=64):
    img  = _base_surface(size)
    draw = ImageDraw.Draw(img)
    for _ in range(random.randint(1, 4)):
        x1, y1 = random.randint(2, size-2), random.randint(2, size-2)
        ang = random.uniform(0, np.pi)
        ln  = random.randint(15, 45)
        x2  = int(x1 + ln * np.cos(ang))
        y2  = int(y1 + ln * np.sin(ang))
        col = random.randint(35, 80)
        draw.line([(x1, y1), (x2, y2)],
                  fill=(col, col, col+5), width=random.randint(1, 3))
    return img

def make_stain(size=64):
    img  = _base_surface(size)
    draw = ImageDraw.Draw(img)
    for _ in range(random.randint(1, 3)):
        cx, cy = random.randint(8, 56), random.randint(8, 56)
        rx, ry = random.randint(5, 18), random.randint(4, 13)
        r = random.randint(90, 165)
        g = random.randint(50, 100)
        b = random.randint(10, 55)
        draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=(r, g, b))
    return img.filter(ImageFilter.GaussianBlur(1.3))

GENERATORS = {'dent': make_dent, 'scratch': make_scratch, 'stain': make_stain}
N_SYNTH = 120

for cls, fn in GENERATORS.items():
    os.makedirs(f'images/{cls}', exist_ok=True)
    for i in range(1, N_SYNTH + 1):
        path = f'images/{cls}/{cls}_{i:03d}.png'
        if not os.path.exists(path):
            fn(IMG_SIZE).save(path)

print('Dataset class summary:')
for cls in CLASS_NAMES:
    n = len(os.listdir(f'images/{cls}'))
    print(f'  {cls:8s}: {n} images')

TOTAL = sum(len(os.listdir(f'images/{c}')) for c in CLASS_NAMES)
print(f'  {"TOTAL":8s}: {TOTAL} images')


In [ ]:
# ── Image dimensions check ────────────────────────────────────────────────────
print('=== Image Dimension Analysis ===')
dims = []
for cls in CLASS_NAMES:
    sample = sorted(os.listdir(f'images/{cls}'))[0]
    img = Image.open(f'images/{cls}/{sample}')
    dims.append({'Class': cls, 'Width': img.size[0], 'Height': img.size[1],
                 'Mode': img.mode, 'Channels': 3 if img.mode=='RGB' else 1})
print(pd.DataFrame(dims).to_string(index=False))

# Class imbalance check
counts = {cls: len(os.listdir(f'images/{cls}')) for cls in CLASS_NAMES}
max_c, min_c = max(counts.values()), min(counts.values())
imbalance_ratio = max_c / min_c
print(f'\nImbalance ratio (max/min): {imbalance_ratio:.2f}x')
print('→ ' + ('Mild imbalance — apply class weights during training.' if imbalance_ratio > 1.5
              else 'Dataset is balanced — no special handling needed.'))


In [ ]:
# ── Visual exploration ────────────────────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(14, 14))
fig.suptitle('Task 2 — Sample Images from Each Class (5 per class)',
             fontsize=14, fontweight='bold')

cls_colors = {'dent':'#EF5350','normal':'#42A5F5','scratch':'#FFA726','stain':'#66BB6A'}

# Row 0: class label cards
for col, cls in enumerate(CLASS_NAMES):
    axes[0, col].set_facecolor(cls_colors[cls])
    axes[0, col].text(0.5, 0.5, cls.upper(), ha='center', va='center',
                      fontsize=14, fontweight='bold', color='white',
                      transform=axes[0, col].transAxes)
    axes[0, col].axis('off')
axes[0, -1].axis('off')

# Rows 1-4: sample images
for row in range(1, 5):
    for col, cls in enumerate(CLASS_NAMES):
        files = sorted(os.listdir(f'images/{cls}'))
        fname = files[(row-1) % len(files)]
        img   = Image.open(f'images/{cls}/{fname}').convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{fname}', fontsize=7)
        axes[row, col].axis('off')
        for spine in axes[row, col].spines.values():
            spine.set_edgecolor(cls_colors[cls])
            spine.set_linewidth(3)
    axes[row, -1].axis('off')

plt.tight_layout()
plt.savefig('results/task2_sample_images.png', bbox_inches='tight')
plt.show()
print('✅ Saved → results/task2_sample_images.png')


In [ ]:
# ── Class distribution bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Task 2 — Class Distribution', fontsize=13, fontweight='bold')

colors = [cls_colors[c] for c in CLASS_NAMES]
bars   = axes[0].bar(CLASS_NAMES, [counts[c] for c in CLASS_NAMES],
                     color=colors, edgecolor='white', linewidth=1.5)
for bar, cls in zip(bars, CLASS_NAMES):
    v = counts[cls]
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1.5, str(v),
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Number of Images'); axes[0].set_title('Image Count per Class')
axes[0].set_ylim(0, max(counts.values())*1.25)

axes[1].pie([counts[c] for c in CLASS_NAMES], labels=CLASS_NAMES,
            autopct='%1.1f%%', colors=colors,
            wedgeprops={'edgecolor':'white','linewidth':2}, startangle=90)
axes[1].set_title('Proportion by Class')

plt.tight_layout()
plt.savefig('results/task2_class_distribution.png', bbox_inches='tight')
plt.show()


---
## 🔧 Task 3 — Image Preprocessing

In [ ]:
# ── Load all images into NumPy arrays ─────────────────────────────────────────
label_map = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}
X_all, y_all = [], []

for cls in CLASS_NAMES:
    folder = f'images/{cls}'
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.png','.jpg','.jpeg')):
            continue
        img = Image.open(f'{folder}/{fname}').convert('RGB')
        # ── Step 1: Resize to fixed 64×64 ──────────────────────────────────
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        X_all.append(np.array(img, dtype=np.float32))
        y_all.append(label_map[cls])

X = np.array(X_all)
y = np.array(y_all)

# ── Step 2: Normalize pixel values to [0, 1] ──────────────────────────────────
X = X / 255.0

print('=== Preprocessing Summary ===')
print(f'Total images  : {len(X)}')
print(f'Image shape   : {X[0].shape}  (H × W × C)')
print(f'Pixel range   : [{X.min():.3f}, {X.max():.3f}]  ← normalised to [0, 1]')
print(f'Data type     : {X.dtype}')
print(f'Total memory  : {X.nbytes / 1e6:.2f} MB')


In [ ]:
# ── Step 3: Train / Validation / Test split ───────────────────────────────────
# 70% train | 15% validation | 15% test
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)

# One-hot encode labels
y_tr_cat  = to_categorical(y_tr,  N_CLASSES)
y_val_cat = to_categorical(y_val, N_CLASSES)
y_te_cat  = to_categorical(y_te,  N_CLASSES)

print('=== Dataset Splits ===')
print(f'Training   : {X_tr.shape[0]} images  ({X_tr.shape[0]/len(X)*100:.0f}%)')
print(f'Validation : {X_val.shape[0]} images  ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test       : {X_te.shape[0]} images  ({X_te.shape[0]/len(X)*100:.0f}%)')

# Verify class distribution maintained
print('\nTrain class distribution:', {CLASS_NAMES[i]: int((y_tr==i).sum()) for i in range(N_CLASSES)})


In [ ]:
# ── Step 4: Data Augmentation ─────────────────────────────────────────────────
# Augmentation artificially increases training diversity to improve generalisation.
# Applied ONLY to training data.

datagen = ImageDataGenerator(
    rotation_range=15,          # Rotate ±15 degrees
    width_shift_range=0.10,     # Shift horizontally ±10%
    height_shift_range=0.10,    # Shift vertically ±10%
    horizontal_flip=True,       # Random horizontal flip
    zoom_range=0.10,            # Zoom in/out ±10%
    brightness_range=[0.85, 1.15],  # Brightness variation
    fill_mode='nearest'
)

# Visualise augmented samples
sample_idx = 5
sample_img = X_tr[sample_idx:sample_idx+1]

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle('Task 3 — Data Augmentation Examples', fontsize=13, fontweight='bold')

axes[0, 0].imshow(sample_img[0]); axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')
for i, ax in enumerate(axes.flat[1:]):
    aug_gen = datagen.flow(sample_img, batch_size=1)
    aug_img = next(aug_gen)[0]
    ax.imshow(np.clip(aug_img, 0, 1))
    ax.set_title(f'Augmented {i+1}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/task3_augmentation.png', bbox_inches='tight')
plt.show()
print('✅ Saved → results/task3_augmentation.png')


---
## 🧠 Task 4 — CNN Model Creation

### Architecture Design

```
Input (64×64×3)
│
├─ [BLOCK 1] Conv2D(32, 3×3) → BN → ReLU → Conv2D(32, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25)
│
├─ [BLOCK 2] Conv2D(64, 3×3) → BN → ReLU → Conv2D(64, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25)
│
├─ [BLOCK 3] Conv2D(128, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25)
│
├─ Flatten
├─ Dense(256, ReLU) → Dropout(0.5)
└─ Dense(4, Softmax)  ← output: probability over 4 classes
```


In [ ]:
def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), n_classes=N_CLASSES):
    """Build a 3-block CNN for vehicle defect classification."""
    model = Sequential(name='VehicleDefectCNN')

    # ── Block 1: detect low-level features (edges, gradients) ──────────────────
    model.add(Conv2D(32, (3,3), padding='same', input_shape=input_shape, name='conv1_1'))
    model.add(BatchNormalization(name='bn1_1'))
    model.add(tf.keras.layers.Activation('relu', name='relu1_1'))
    model.add(Conv2D(32, (3,3), padding='same', name='conv1_2'))
    model.add(BatchNormalization(name='bn1_2'))
    model.add(tf.keras.layers.Activation('relu', name='relu1_2'))
    model.add(MaxPooling2D((2,2), name='pool1'))   # 64→32
    model.add(Dropout(0.25, name='drop1'))

    # ── Block 2: detect mid-level patterns (curves, textures) ──────────────────
    model.add(Conv2D(64, (3,3), padding='same', name='conv2_1'))
    model.add(BatchNormalization(name='bn2_1'))
    model.add(tf.keras.layers.Activation('relu', name='relu2_1'))
    model.add(Conv2D(64, (3,3), padding='same', name='conv2_2'))
    model.add(BatchNormalization(name='bn2_2'))
    model.add(tf.keras.layers.Activation('relu', name='relu2_2'))
    model.add(MaxPooling2D((2,2), name='pool2'))   # 32→16
    model.add(Dropout(0.25, name='drop2'))

    # ── Block 3: detect high-level damage signatures ────────────────────────────
    model.add(Conv2D(128, (3,3), padding='same', name='conv3_1'))
    model.add(BatchNormalization(name='bn3_1'))
    model.add(tf.keras.layers.Activation('relu', name='relu3_1'))
    model.add(MaxPooling2D((2,2), name='pool3'))   # 16→8
    model.add(Dropout(0.25, name='drop3'))

    # ── Classifier head ────────────────────────────────────────────────────────
    model.add(Flatten(name='flatten'))
    model.add(Dense(256, activation='relu', name='fc1'))
    model.add(Dropout(0.50, name='drop_fc'))
    model.add(Dense(n_classes, activation='softmax', name='output'))  # softmax → probabilities

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


tf.random.set_seed(SEED)
cnn = build_cnn()
cnn.summary()
print(f'\nTotal parameters: {cnn.count_params():,}')


---
## 🏋️ Task 5 — Model Training and Evaluation

In [ ]:
# ── Training callbacks ─────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=12,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=6, min_lr=1e-6, verbose=1)
]

# ── Use augmented generator for training ───────────────────────────────────────
BATCH_SIZE = 32
train_gen  = datagen.flow(X_tr, y_tr_cat, batch_size=BATCH_SIZE, seed=SEED)

print('Training CNN model...')
history = cnn.fit(
    train_gen,
    steps_per_epoch=len(X_tr) // BATCH_SIZE,
    epochs=80,
    validation_data=(X_val, y_val_cat),
    callbacks=callbacks,
    verbose=1
)
print(f'\nStopped at epoch {len(history.history["loss"])}')


In [ ]:
# ── Accuracy and Loss curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Task 5 — Training History', fontsize=14, fontweight='bold')
ep = range(1, len(history.history['loss'])+1)

axes[0].plot(ep, history.history['loss'],     'b-o', ms=3, lw=2, label='Train Loss')
axes[0].plot(ep, history.history['val_loss'], 'r-o', ms=3, lw=2, label='Val Loss')
best = np.argmin(history.history['val_loss'])+1
axes[0].axvline(x=best, color='green', ls='--', alpha=0.7, label=f'Best epoch {best}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Categorical Cross-Entropy Loss')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].grid(True, alpha=.3)

axes[1].plot(ep, history.history['accuracy'],     'b-o', ms=3, lw=2, label='Train Acc')
axes[1].plot(ep, history.history['val_accuracy'], 'r-o', ms=3, lw=2, label='Val Acc')
axes[1].axvline(x=best, color='green', ls='--', alpha=0.7, label=f'Best epoch {best}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve'); axes[1].legend(); axes[1].grid(True, alpha=.3)

plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', bbox_inches='tight')
plt.show()
print('✅ Saved → results/accuracy_loss_curves.png')


In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────────
test_loss, test_acc = cnn.evaluate(X_te, y_te_cat, verbose=0)
y_pred_prob = cnn.predict(X_te, verbose=0)
y_pred      = y_pred_prob.argmax(axis=1)
y_true      = y_te

print('=== Test Set Performance ===')
print(f'  Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  Test Loss     : {test_loss:.4f}')
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
annot  = np.array([[f'{v}\n({p:.1f}%)' for v,p in zip(rv,rp)]
                   for rv,rp in zip(cm, cm_pct)])

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=annot, fmt='', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=1, linecolor='white',
            cbar_kws={'shrink':0.8})
plt.title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Actual Class', fontweight='bold')
plt.xlabel('Predicted Class', fontweight='bold')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', bbox_inches='tight')
plt.show()
print('✅ Saved → results/confusion_matrix.png')


In [ ]:
# ── Sample predictions ────────────────────────────────────────────────────────
n_show = 20
indices = random.sample(range(len(X_te)), n_show)

fig, axes = plt.subplots(4, 5, figsize=(14, 11))
fig.suptitle('Task 5 — Sample Predictions on Test Images\n(Green border = Correct  |  Red border = Wrong)',
             fontsize=13, fontweight='bold')

for ax, idx in zip(axes.flat, indices):
    img = X_te[idx]
    true_cls = CLASS_NAMES[y_true[idx]]
    pred_cls = CLASS_NAMES[y_pred[idx]]
    conf     = y_pred_prob[idx].max()
    correct  = true_cls == pred_cls

    ax.imshow(img)
    ax.axis('off')

    # Colored border
    border_color = '#43A047' if correct else '#E53935'
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_color)
        spine.set_linewidth(4)

    label = f'✓ {pred_cls}\n{conf:.0%}' if correct else f'✗ {pred_cls} ({conf:.0%})\nTrue: {true_cls}'
    ax.set_title(label, fontsize=8.5, fontweight='bold',
                 color='#43A047' if correct else '#E53935')

plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', bbox_inches='tight')
plt.show()
print('✅ Saved → sample_predictions/prediction_outputs.png')


In [ ]:
# ── Per-class accuracy breakdown ──────────────────────────────────────────────
per_class_acc = {CLASS_NAMES[i]: (y_pred[y_true==i]==i).mean()
                 for i in range(N_CLASSES)}

plt.figure(figsize=(8, 4))
bar_cols = [cls_colors[c] for c in CLASS_NAMES]
bars = plt.bar(CLASS_NAMES, [per_class_acc[c]*100 for c in CLASS_NAMES],
               color=bar_cols, edgecolor='white', linewidth=1.5)
plt.axhline(y=test_acc*100, color='gray', ls='--', lw=2,
            label=f'Overall Test Acc: {test_acc*100:.1f}%')
for bar, cls in zip(bars, CLASS_NAMES):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{per_class_acc[cls]*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.ylabel('Accuracy (%)'); plt.ylim(0, 115)
plt.title('Per-Class Test Accuracy', fontweight='bold', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig('results/per_class_accuracy.png', bbox_inches='tight')
plt.show()

print('\nPer-class accuracies:')
for cls, acc in per_class_acc.items():
    print(f'  {cls:8s}: {acc:.2%}')


In [ ]:
# ── Feature map visualisation ─────────────────────────────────────────────────
# Show what the first conv layer 'sees'
activation_model = tf.keras.Model(
    inputs=cnn.input,
    outputs=cnn.get_layer('conv1_1').output
)
sample = X_te[0:1]
feature_maps = activation_model.predict(sample, verbose=0)  # (1, H, W, 32)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('Conv Layer 1 Feature Maps (32 filters) — What the CNN Sees',
             fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < feature_maps.shape[-1]:
        ax.imshow(feature_maps[0, :, :, i], cmap='viridis')
    ax.axis('off')
plt.tight_layout()
plt.savefig('results/feature_maps.png', bbox_inches='tight')
plt.show()
print('✅ Saved → results/feature_maps.png')


---
## 🔬 Task 6 — CNN Concept Explanation

*(See README.md for the full written explanation. Summary below.)*

### What is Convolution?
A **learnable filter** (kernel) of size 3×3 or 5×5 slides across the entire image, computing the dot product at each position. This produces a **feature map** that highlights where a specific pattern (edge, curve, colour gradient) appears in the image.

$$\text{feature\_map}(i, j) = \sum_{m}\sum_{n} \text{input}(i+m,\, j+n) \times \text{filter}(m,\, n) + b$$

The filter weights are **learned from data** — the network automatically discovers which patterns are diagnostic for each class.

### Why is Pooling Used?
**MaxPooling** takes the maximum value in each `k×k` region, reducing spatial size by factor `k`. Benefits:
- **Dimensionality reduction**: fewer parameters → faster training
- **Translation invariance**: a dent slightly left or right of centre is still detected
- **Hierarchical abstraction**: deeper layers see larger receptive fields

### Why is ReLU Commonly Used?
- **No vanishing gradient**: unlike sigmoid/tanh, ReLU keeps gradients from shrinking in deep networks
- **Sparsity**: setting negatives to 0 creates sparse activations → efficient computation
- **Simple and fast**: `max(0, x)` — trivial to compute and differentiate

### Why CNNs Beat Regular Neural Networks for Images?
| Property | Feed-Forward NN | CNN |
|---|---|---|
| **Parameters** | 64×64×3 = 12,288 inputs × hidden neurons = millions | Filters shared across all positions → ~32×3×3×3 ≈ 864 params |
| **Local patterns** | Treats every pixel independently | Filters detect local structure (edges, corners) |
| **Translation** | Sensitive to object position | Pooling provides translation invariance |
| **Hierarchy** | No built-in hierarchy | Low → mid → high-level features automatically |


---
## 💼 Task 7 — Business Use Case Mapping

### Domain: Manufacturing — Automotive Quality Control

#### Problem
On a high-throughput vehicle assembly line, hundreds of vehicles pass through the paint and body shop daily. Currently, **10–15 human inspectors** visually check each vehicle for surface defects. This process is:
- **Slow** (~8 minutes per vehicle)
- **Inconsistent** (inspector fatigue after 4 hours)
- **Expensive** (~₹85L/year in inspector salaries)

#### AI Solution
Deploy ceiling-mounted cameras at the end of the assembly line. A real-time CNN model classifies each vehicle image as `normal` / `dent` / `scratch` / `stain` within **200 ms**.

| Defect Class | Automated Response |
|---|---|
| `normal` | Vehicle proceeds to dispatch |
| `dent` | Routed to dent-repair station |
| `scratch` | Routed to re-spray cabin |
| `stain` | Routed to detail-cleaning bay |

#### Expected Business Impact
| Metric | Before AI | After AI |
|---|---|---|
| Inspection time per car | 8 min | 0.2 sec |
| Defect escape rate | 4.5% | <0.8% |
| Inspector headcount | 12 | 3 (auditors only) |
| Annual saving | — | ~₹65L |

#### Other Domains
- **Insurance**: Automatically triage vehicle damage photos uploaded during claims
- **Healthcare**: Classify skin lesions (normal vs. melanoma vs. benign)
- **Agriculture**: Detect crop disease types from field drone imagery
- **Retail**: Identify shelf stocking status from store cameras


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=' * 55)
print('  PART 2 — FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  Problem Type   : Multi-Class Image Classification')
print(f'  Classes        : {CLASS_NAMES}')
print(f'  Total Images   : {TOTAL}')
print(f'  Image Size     : {IMG_SIZE}×{IMG_SIZE}×3')
print(f'  CNN Parameters : {cnn.count_params():,}')
print(f'  Test Accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  Test Loss      : {test_loss:.4f}')
print('=' * 55)
print('\nResult files:')
for f in sorted(os.listdir('results')):
    print(f'  ✅ results/{f}')
for f in sorted(os.listdir('sample_predictions')):
    print(f'  ✅ sample_predictions/{f}')
